# Reading the customers

**Lecture 21 · Build** · Géron, Chapter 14

Applications of Machine Learning — BSc Mathematics of Artificial Intelligence

---

**How to use this notebook.** You are not expected to type the code. You are
expected to *read* it before you run it, and to be able to say what every line
does and what would break if it changed. Cells marked **⚠ read before running**
contain a defect on purpose.

**Scale.** The lecture's numbers come from 20,000 training reviews and four
epochs — about ten minutes per run on a GPU. Here we use **5,000 reviews and
two epochs** so the whole notebook finishes in a few minutes. The accuracies are
lower than the deck's; the *ordering* of the four configurations is the same,
and the ordering is the point.

**About the prompt boxes.** Every code cell in this notebook is preceded by a
quoted prompt, and three lines follow it: what the prompt leaves open, the
version a student typically writes instead, and how you would catch a wrong
answer. Those three lines are the part worth reading twice.

The prompts here are **specifications, not transcripts** — this is what you
would have to ask for in order to get this cell, not a recording of somebody
asking for it. If your own prompt is vaguer than the box, expect worse code than
the cell below it.

*Lecture 19 is the one exception in this course.* It was built cell by cell
against Colab's Gemini 3.1 Pro, and its prompts are verbatim. It says so itself.


## 1 · Setup

> **Prompt · setup**
>
> **input** · nothing
>
> **output** · versions, seeds, device
>
> **constraint** · say what to do if there is no accelerator — the GRU cells here are the slowest in the course on CPU

**Watch this prompt.**

* **Left open:** the scale note. The deck's numbers come from 20,000 reviews and four epochs; this notebook uses 5,000 and two, so the accuracies are lower and the ORDERING of the four configurations is the same.
* **The usual student version:** comparing a number from this notebook against one from the slides and concluding something is broken.
* **How you would catch it:** when a notebook is deliberately smaller than the deck, say by how much and say what is preserved. 'The ordering is the point' is a claim you can check.

In [ ]:
# --- setup -------------------------------------------------------------------
# Not examinable: engineering hygiene. It is here because a version mismatch
# produces a confusing error twenty cells later.
import sys, re, time, tarfile, urllib.request
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import sklearn

print(f"python        {sys.version.split()[0]}")
print(f"torch         {torch.__version__}")
print(f"scikit-learn  {sklearn.__version__}")

RANDOM_STATE = 42                 # every split, every model, every shuffle
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"\ndevice        {device}")
if device == "cpu":
    print("No accelerator. Everything below still runs; the GRU cells are slow.")
    print("In Colab: Runtime -> Change runtime type -> T4 GPU.")

## 2 · The corpus

A function, not a manual download — the same rule as Lecture 1. About 80 MB;
**⏱ 30–90 seconds** the first time, instant afterwards.

The split into `train` and `test` ships with the corpus. We do not make our own.

> **Prompt · ⏱ 30-90 s first time — the corpus**
>
> **input** · the IMDb tarball
>
> **output** · 25,000 training and 25,000 test reviews with their labels
>
> **constraint** · use the split that SHIPS with the corpus — we do not make our own, because every published number on this dataset uses that cut
>
> **check** · assert both halves are 25,000 and that the labels are 0/1

**Watch this prompt.**

* **Left open:** that both halves are balanced by construction. That is what makes accuracy a defensible metric here, and it is the condition application 2 spent ninety minutes on.
* **The usual student version:** pooling the two halves and re-splitting, which gives the right sizes and makes the result incomparable with the literature.
* **How you would catch it:** print the positive share of BOTH halves. Balanced-by-construction is a claim about the file, and it costs one line to verify.

In [ ]:
URL  = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
ROOT = Path("datasets")
DATA = ROOT / "aclImdb"

def load_imdb():
    if not DATA.is_dir():
        ROOT.mkdir(parents=True, exist_ok=True)
        tarball = ROOT / "aclImdb_v1.tar.gz"
        if not tarball.is_file():
            urllib.request.urlretrieve(URL, tarball)
        with tarfile.open(tarball) as t:
            t.extractall(path=ROOT, filter="data")

    def read(split):
        texts, labels = [], []
        for lab, name in ((1, "pos"), (0, "neg")):
            for p in sorted((DATA / split / name).iterdir()):
                texts.append(p.read_text(encoding="utf-8"))
                labels.append(lab)
        return texts, np.array(labels, dtype=np.int64)

    return read("train"), read("test")

(train_x, train_y), (test_x, test_y) = load_imdb()

assert len(train_x) == 25_000 and len(test_x) == 25_000, "unexpected corpus size"
assert set(train_y) == {0, 1}
print(f"{len(train_x):,} train, {len(test_x):,} test")
print(f"positive share: train {train_y.mean():.3f}, test {test_y.mean():.3f}")

Balanced by construction, in both halves. That is what makes accuracy a
defensible metric here — the condition Lecture 4 spent ninety minutes on.

Now carve a validation set out of the **training** half. The test half is not
touched again until the very last cell.

> **Prompt · carve the validation set out of TRAINING**
>
> **input** · the 25,000 training reviews
>
> **output** · 5,000 fit and 2,000 validation reviews
>
> **constraint** · both from the TRAINING half — the test half is not touched again until the very last cell
>
> **check** · assert the two index sets are disjoint and the sizes are right

**Watch this prompt.**

* **Left open:** that the corpus ships positives first, so any unshuffled prefix is all one class. The permutation is what makes the slice legitimate.
* **The usual student version:** `train_x[:5000]`, which is 5,000 positive reviews and zero negative ones. The model then learns to answer 'positive' and the validation set agrees with it.
* **How you would catch it:** whenever you slice a corpus, ask whether it is sorted by label. This one is, and so are most of the classic text datasets.

In [ ]:
N_FIT, N_VAL = 5_000, 2_000        # the deck uses 20,000 and 5,000

rng   = np.random.default_rng(RANDOM_STATE)
order = rng.permutation(len(train_x))
fit_i, val_i = order[:N_FIT], order[N_FIT:N_FIT + N_VAL]

fit_x = [train_x[i] for i in fit_i]; fit_y = train_y[fit_i]
val_x = [train_x[i] for i in val_i]; val_y = train_y[val_i]

assert set(fit_i).isdisjoint(val_i), "the split overlaps"
assert len(fit_x) == N_FIT and len(val_x) == N_VAL
print(f"fit {len(fit_x):,}   val {len(val_x):,}   test {len(test_x):,} (untouched)")

## 3 · Look at it — at the training half only

Two things to notice: how long a review is, and how many distinct words there
are. Both decide something about the model.

> **Prompt · look at it — lengths first**
>
> **input** · the training reviews
>
> **output** · the length distribution, and what a cut at MAXLEN costs
>
> **constraint** · report BOTH what fraction of REVIEWS get truncated and what fraction of WORDS survive — they are very different numbers

**Watch this prompt.**

* **Left open:** that three decisions are already made inside the tokenizer: lowercasing, the `<br />` replacement, and the character class that defines a word. None of them is neutral.
* **The usual student version:** choosing MAXLEN by looking at the mean. The distribution has a long tail, and the mean is not where you should cut.
* **How you would catch it:** a truncation length is a hyperparameter. State what it discards, in both units, before adopting it.

In [ ]:
WORD_RE = re.compile(r"[a-z0-9']+")

def word_tokens(s):
    """The whole tokenizer. Three decisions are already made in it."""
    return WORD_RE.findall(s.lower().replace("<br />", " "))

train_tok = [word_tokens(s) for s in train_x]
lens = np.array([len(t) for t in train_tok])

print(f"length: mean {lens.mean():.0f}   median {np.median(lens):.0f}   "
      f"90th pct {np.percentile(lens, 90):.0f}   max {lens.max()}")

MAXLEN = 192                       # the deck uses 256
over = (lens > MAXLEN).mean()
kept = np.minimum(lens, MAXLEN).sum() / lens.sum()
print(f"cutting at {MAXLEN}: truncates {over:.1%} of reviews, "
      f"keeps {kept:.1%} of all words")

> **Prompt · and the vocabulary**
>
> **input** · every token in the training half
>
> **output** · how many distinct words, how many appear exactly once, and the length histogram
>
> **constraint** · count the HAPAX words — the ones seen exactly once — as a share of the vocabulary

**Watch this prompt.**

* **Left open:** what follows: two words in five are seen once and never again. You cannot learn a 128-dimensional vector for a word from one example, and that is the diagnosis the whole lecture builds to.
* **The usual student version:** seeing 100,000 distinct words and concluding the model needs a bigger vocabulary. Most of that vocabulary is unlearnable at any size.
* **How you would catch it:** clip the histogram before plotting. One review of 2,470 words stretches the axis so that the bulk of the distribution is three pixels wide.

In [ ]:
counts = Counter(w for t in train_tok for w in t)
hapax  = sum(1 for _, c in counts.items() if c == 1)
print(f"{len(counts):,} distinct words")
print(f"{hapax:,} of them appear exactly once  ({hapax / len(counts):.1%})")

plt.figure(figsize=(9, 3))
plt.hist(np.clip(lens, 0, 1200), bins=80)
plt.axvline(MAXLEN, color="C3", ls="--")
plt.xlabel("review length, in words"); plt.ylabel("reviews")
plt.tight_layout(); plt.show()

Two words in five are seen once and never again. You cannot learn a
128-dimensional vector for a word from one example — remember that when the
recurrent model underperforms.

## 4 · The baselines, before anything is built

Rule 2 of this course: *a metric with nothing to compare it to is decoration.*

Two anchors. The trivial one, and the one that is actually hard to beat.

> **Prompt · the trivial anchor**
>
> **input** · the test labels
>
> **output** · the majority-class accuracy
>
> **constraint** · take the max of the two shares rather than assuming which class is commoner

**Watch this prompt.**

* **Left open:** that this anchor is nearly useless here. The corpus is balanced, so it sits at 50%, and the interesting anchor is the next cell.
* **The usual student version:** stopping at this baseline. Beating 50% on a balanced binary task is not evidence of anything.
* **How you would catch it:** a trivial baseline that is trivially beaten still belongs in the table. It is the zero of the scale.

In [ ]:
majority = max(test_y.mean(), 1 - test_y.mean())
print(f"always predict one class:  {majority:.1%}")

> **Prompt · ⏱ 15 s — the anchor that is actually hard to beat**
>
> **input** · all 25,000 training reviews, unigrams and bigrams
>
> **output** · a tf-idf logistic regression and its test accuracy
>
> **constraint** · `fit_transform` on train and `transform` on test — different verbs, and the vocabulary is a fitted object like any other
>
> **check** · assert the two matrices have the same number of COLUMNS, which is what fails if `fit_transform` was called twice

**Watch this prompt.**

* **Left open:** that this is the number to estimate against in the commitment exercise, not the 50%. Counting words is a strong baseline on sentiment.
* **The usual student version:** `vec.fit_transform(test_x)` on the second line, which builds a different feature space, and then a shape error much later — or worse, no error at all if the counts happen to match.
* **How you would catch it:** thirty seconds of counting words is the thing your recurrent network has to beat. Measure it before you build anything.

In [ ]:
# ⏱ about 15 seconds: 25,000 documents, unigrams and bigrams.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

t0 = time.perf_counter()
vec = TfidfVectorizer(min_df=2, ngram_range=(1, 2), max_features=200_000)
X_bow_train = vec.fit_transform(train_x)          # fit on TRAINING reviews only
X_bow_test  = vec.transform(test_x)               # transform, a different verb

bow = LogisticRegression(max_iter=2000, C=4.0).fit(X_bow_train, train_y)
bow_acc = (bow.predict(X_bow_test) == test_y).mean()

assert X_bow_train.shape[0] == 25_000
assert X_bow_train.shape[1] == X_bow_test.shape[1], "different feature spaces"
print(f"tf-idf + logistic regression: {bow_acc:.1%}  "
      f"({X_bow_train.shape[1]:,} features, {time.perf_counter() - t0:.0f}s)")

## 5 · Commit

**Stop. On paper, now.** Not in this notebook — on paper, where you cannot
quietly revise it.

```
Metric:                                            ____________
Accuracy the desk would need to trust it:        ____________ %
Accuracy I expect from the model I build today:  ____________ %
Best accuracy I actually obtain:                 ____________ %
```

You have two anchors: the trivial one and the bag of words. Estimate against the
second, not the first.

## 6 · Tokenisation, and why a word vocabulary breaks

Build the vocabulary from the **fit split only**. Building a vocabulary is
*fitting*, and it obeys the same rule as every other fitted object in this
course.

> **Prompt · the vocabulary, fitted on the FIT split only**
>
> **input** · the 5,000 fit reviews
>
> **output** · a word-to-index map, and the out-of-vocabulary rate on test
>
> **constraint** · build it from the fit split ONLY — building a vocabulary is FITTING, and it obeys the same rule as every other fitted object in this course
>
> **check** · assert indices 0 and 1 are reserved, for padding and [UNK]

**Watch this prompt.**

* **Left open:** the two OOV rates. The rate over TOKENS looks survivable; the rate over DISTINCT words never does — and the distinct words are the informative ones.
* **The usual student version:** building the vocabulary from all 25,000 training reviews, or from train plus test. The second is leakage; the first is subtler and still makes the validation OOV rate optimistic.
* **How you would catch it:** the problem is not the SIZE of the vocabulary. A word vocabulary is closed and language is not, so no size fixes it.

In [ ]:
VOCAB = 20_000
fit_counts = Counter(w for s in fit_x for w in word_tokens(s))
w2i = {w: i + 2 for i, (w, _) in enumerate(fit_counts.most_common(VOCAB - 2))}
# 0 = padding, 1 = [UNK]

assert 0 not in w2i.values() and 1 not in w2i.values()
assert len(w2i) <= VOCAB - 2
print(f"{len(w2i):,} words in the vocabulary")

test_flat = [w for s in test_x for w in word_tokens(s)]
oov = sum(1 for w in test_flat if w not in w2i)
distinct_test = set(test_flat)
oov_types = sum(1 for w in distinct_test if w not in w2i)
print(f"unseen test tokens:         {oov / len(test_flat):.1%}")
print(f"unseen DISTINCT test words: {oov_types / len(distinct_test):.1%}")

The token rate looks survivable. The rate over *distinct* words never does — and
those are the informative ones. The problem is not the size of the vocabulary:
a word vocabulary is **closed** and language is not.

Now the same sentence under a subword tokenizer, which was trained once on some
other corpus and shipped.

> **Prompt · the same sentence, under subwords**
>
> **input** · one sentence with two rare words in it
>
> **output** · the word tokenizer's view beside WordPiece's
>
> **constraint** · show the word tokenizer producing [UNK] for exactly the words that carry the sentiment

**Watch this prompt.**

* **Left open:** that WordPiece was trained once on some other corpus and shipped. It is not adapted to IMDb and does not need to be.
* **The usual student version:** reading 'discombobulating' as [UNK] and shrugging. It is the most informative word in the sentence and the model never sees it.
* **How you would catch it:** print the pieces-per-word ratio and both vocabulary sizes. A subword tokenizer buys coverage and spends sequence length, and the next sections are about that trade.

In [ ]:
from transformers import AutoTokenizer

BERT = "distilbert-base-uncased"
tk = AutoTokenizer.from_pretrained(BERT)

sentence = "The plot was unwatchable and utterly discombobulating."
words  = word_tokens(sentence)
pieces = tk.tokenize(sentence)

print("word tokenizer :", [w if w in w2i else "[UNK]" for w in words])
print("WordPiece      :", pieces)
print(f"\n{len(words)} words -> {len(pieces)} pieces "
      f"({len(pieces) / len(words):.2f} per word)")
print(f"vocabulary: ours {VOCAB:,}   WordPiece {tk.vocab_size:,}")

> **Prompt · how often does WordPiece give up**
>
> **input** · 2,000 randomly chosen test reviews
>
> **output** · the [UNK] rate over subword tokens
>
> **constraint** · sample RANDOMLY — the corpus ships positives first, so `test_x[:2000]` is all positives
>
> **check** · assert the rate is below 0.1%, which is what 'almost never misses' should mean

**Watch this prompt.**

* **Left open:** that the label does not matter for a rate that ignores it. The comment says so and shuffles anyway, because the habit is worth more than the exception.
* **The usual student version:** taking a prefix, on the grounds that the measurement does not depend on the label. It does not here, and it will next time.
* **How you would catch it:** measure rather than assume. 'Subword tokenizers have no OOV problem' is nearly true and the nearly is worth four lines.

In [ ]:
# How often does WordPiece have to give up? Measure rather than assume.
# NB: the corpus ships positives first, so test_x[:2000] is all positives. It
# does not matter for a rate that ignores the label, but take the habit anyway.
sample_i = np.random.default_rng(RANDOM_STATE + 7).permutation(len(test_x))[:2_000]
sample = [test_x[i] for i in sample_i]
n_unk = n_tok = 0
for s in sample:
    ids = tk(s, truncation=True, max_length=512)["input_ids"]
    n_unk += sum(1 for i in ids if i == tk.unk_token_id)
    n_tok += len(ids)
print(f"[UNK] rate over {n_tok:,} subword tokens: {n_unk / n_tok:.4%}")
assert n_unk / n_tok < 0.001, "a subword tokenizer should almost never miss"

## 7 · From integers to vectors

Token 4,271 is not four thousand of anything — the integer is a name. An
embedding is a learned table with one row per token, and looking up row *i* is
exactly multiplying a one-hot vector by that table, without ever forming it.

`padding_idx=0` pins row 0 at zero and keeps it there. Padding must not learn
anything.

> **Prompt · from integers to vectors**
>
> **input** · a batch of token ids
>
> **output** · the embedded batch, and the table's parameter count
>
> **constraint** · `padding_idx=0` — it pins row 0 at zero and KEEPS it there, so padding never learns anything
>
> **check** · assert the output shape, and assert row 0 is still exactly zero

**Watch this prompt.**

* **Left open:** that token 4,271 is not four thousand of anything. The integer is a name, and looking up row i is exactly multiplying a one-hot vector by the table without ever forming it.
* **The usual student version:** omitting `padding_idx`, so the padding row acquires a gradient and drifts. Nothing errors and the padding slowly starts to mean something.
* **How you would catch it:** assert the padding row is zero AFTER training too, not just at construction. That is the only way to know the flag did what it claims.

In [ ]:
emb = nn.Embedding(num_embeddings=VOCAB, embedding_dim=128, padding_idx=0)
x   = torch.randint(0, VOCAB, (32, MAXLEN))

assert emb(x).shape == (32, MAXLEN, 128)
assert torch.equal(emb.weight[0], torch.zeros(128)), "padding row is not zero"
print(f"embedding table: {emb.weight.numel():,} parameters")
print(f"one batch: {tuple(x.shape)} -> {tuple(emb(x).shape)}")

## 8 · Padding, and the length you must keep

A batch is a rectangle; reviews are not. Pad to `MAXLEN`, truncate what is
longer, and **keep the true lengths**. A padded batch has lost the information
about where each review ends.

> **Prompt · padding, and the length you must keep**
>
> **input** · variable-length token sequences
>
> **output** · a rectangular batch and the true lengths
>
> **constraint** · keep the LENGTHS. A padded batch has lost the information about where each review ends, and the model needs it back
>
> **check** · assert every length is between 1 and MAXLEN, and that nothing is written past the true length of the first row

**Watch this prompt.**

* **Left open:** the `max(len(s), 1)` — a length of zero would crash the packing, and an empty review after tokenisation is not impossible.
* **The usual student version:** padding and not recording lengths, which is the setup for the assistant failure two sections down. The batch looks fine and the model reads padding.
* **How you would catch it:** assert that the region past the true length is zero. It is one line and it is the invariant the whole padding scheme depends on.

In [ ]:
def pad_batch(seqs):
    X = np.zeros((len(seqs), MAXLEN), dtype=np.int64)
    L = np.zeros(len(seqs), dtype=np.int64)
    for i, s in enumerate(seqs):
        s = s[:MAXLEN]
        X[i, :len(s)] = s
        L[i] = max(len(s), 1)          # a length of 0 would crash the packing
    return torch.from_numpy(X), torch.from_numpy(L)

def encode_words(texts):
    return pad_batch([[w2i.get(w, 1) for w in word_tokens(s)] for s in texts])

Xf_w, Lf_w = encode_words(fit_x)
Xv_w, Lv_w = encode_words(val_x)

assert Xf_w.shape == (N_FIT, MAXLEN)
assert (Lf_w >= 1).all() and (Lf_w <= MAXLEN).all()
assert (Xf_w[0, Lf_w[0]:] == 0).all(), "something is written past the true length"
print(f"fit batch {tuple(Xf_w.shape)}   lengths {Lf_w.min()}–{Lf_w.max()}")

## 9 · The classifier

Thirteen lines, and eleven of them are Lecture 12. The two new ones are the
embedding and the packing.

> **Prompt · the classifier**
>
> **input** · a padded batch and its lengths
>
> **output** · two logits per review
>
> **constraint** · `pack_padded_sequence` with `enforce_sorted=False`, and take the final hidden state of BOTH directions
>
> **check** · assert the head returns exactly two logits for a batch of four

**Watch this prompt.**

* **Left open:** that eleven of these thirteen lines are the earlier PyTorch lecture. The two new ones are the embedding and the packing.
* **The usual student version:** `bidirectional=True` and then taking `h[-1]`, which is one direction only. The two directions are separate rows of `h` and both are needed.
* **How you would catch it:** `lengths.cpu()` in the pack call. It requires a CPU tensor and the error message when it is on the GPU names neither the argument nor the fix.

In [ ]:
EMB_DIM, HIDDEN = 128, 64

class GRUClassifier(nn.Module):
    def __init__(self, vocab, init=None, freeze=False, last_of_padding=False):
        super().__init__()
        self.emb = nn.Embedding(vocab, EMB_DIM, padding_idx=0)
        if init is not None:
            self.emb.weight.data.copy_(torch.from_numpy(init))
            self.emb.weight.requires_grad = not freeze
        self.rnn  = nn.GRU(EMB_DIM, HIDDEN, batch_first=True, bidirectional=True)
        self.head = nn.Linear(2 * HIDDEN, 2)
        self.last_of_padding = last_of_padding

    def forward(self, x, lengths):
        e = self.emb(x)
        if self.last_of_padding:                 # the assistant's version
            out, _ = self.rnn(e)
            return self.head(out[:, -1, :])
        packed = nn.utils.rnn.pack_padded_sequence(
            e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.rnn(packed)
        return self.head(torch.cat([h[0], h[1]], dim=1))

_m = GRUClassifier(VOCAB)
assert _m(Xf_w[:4], Lf_w[:4]).shape == (4, 2), "the head must return two logits"
print(f"{sum(p.numel() for p in _m.parameters()):,} parameters")

The training loop is Lecture 12's five lines, unchanged. Nothing about text
changes the loop.

⏱ **about 40–90 seconds per run** on a GPU, several minutes on a CPU.

> **Prompt · the loop, with early stopping**
>
> **input** · the training and validation splits
>
> **output** · a trained model, keeping the weights from the BEST validation epoch
>
> **constraint** · count accuracy over the SET, not as a mean of batch accuracies — the entry from application 6

**Watch this prompt.**

* **Left open:** why early stopping is not optional here. Reporting the last epoch instead is worth several points to whichever run overfits hardest, and that is the run with the pretrained embeddings, because it starts from vectors that already mean something.
* **The usual student version:** reporting the final epoch. The comparison at the end is between four configurations, and the one that benefits most from the bug is the one the lecture is arguing for.
* **How you would catch it:** clone the state dict when you save it. `net.state_dict()` returns references to live tensors, so without the clone your 'best' weights keep training.

In [ ]:
BATCH, EPOCHS, LR = 64, 2, 1e-3     # the deck uses 4 epochs

@torch.no_grad()
def accuracy(net, X, L, y, batch=256):
    net.eval()
    hits = 0
    for i in range(0, len(X), batch):
        out = net(X[i:i + batch].to(device), L[i:i + batch])
        hits += int((out.argmax(1).cpu().numpy() == y[i:i + batch]).sum())
    return hits / len(y)            # counted over the SET, not a mean of batches

def train(net, Xf, Lf, yf, Xv, Lv, yv, double_softmax=False, tag=""):
    """Train, and keep the weights from the best validation epoch.

    Early stopping, from Lecture 6. Reporting the last epoch instead is worth
    several points to whichever run overfits hardest — and the run that
    overfits hardest is the one with the pretrained embeddings, because it
    starts from vectors that already mean something.
    """
    net = net.to(device)
    params = [p for p in net.parameters() if p.requires_grad]
    opt    = torch.optim.Adam(params, lr=LR)
    lossf  = nn.CrossEntropyLoss()
    Xf_d, yf_d = Xf.to(device), torch.from_numpy(yf).to(device)
    curve, losses = [], []
    best_state, best_val, best_epoch = None, -1.0, 0
    t0 = time.perf_counter()
    for ep in range(EPOCHS):
        net.train()
        perm, running = torch.randperm(len(Xf_d)), 0.0
        for i in range(0, len(Xf_d), BATCH):
            j = perm[i:i + BATCH]
            opt.zero_grad()
            out = net(Xf_d[j], Lf[j])
            if double_softmax:
                out = torch.softmax(out, dim=1)
            loss = lossf(out, yf_d[j])
            loss.backward()
            opt.step()
            running += float(loss.detach()) * len(j)
        losses.append(running / len(Xf_d))
        curve.append(accuracy(net, Xv, Lv, yv))
        if curve[-1] > best_val:
            best_val, best_epoch = curve[-1], ep + 1
            best_state = {k: v.detach().cpu().clone()
                          for k, v in net.state_dict().items()}
        print(f"  {tag} epoch {ep + 1}: loss {losses[-1]:.4f}  "
              f"val {curve[-1]:.4f}  ({time.perf_counter() - t0:.0f}s)")
    net.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    print(f"  {tag}: keeping epoch {best_epoch} (val {best_val:.4f})")
    return net, curve, losses

> **Prompt · ⏱ 40-90 s — our words, random table**
>
> **input** · the word-tokenised batches
>
> **output** · the trained model and its validation curve
>
> **constraint** · re-seed immediately before, so every configuration below starts from the same initialisation

**Watch this prompt.**

* **Left open:** that this is the configuration the diagnosis is about. The embedding table is being asked to learn English from 5,000 sentiment labels.
* **The usual student version:** running the four configurations without re-seeding between them, so part of every difference is initialisation.
* **How you would catch it:** `torch.manual_seed` before EACH run, not once at the top. Any cell that consumes randomness in between shifts every configuration after it.

In [ ]:
torch.manual_seed(RANDOM_STATE)
scratch, curve_scratch, loss_scratch = train(
    GRUClassifier(VOCAB), Xf_w, Lf_w, fit_y, Xv_w, Lv_w, val_y,
    tag="our words / random")

## 10 · ⚠ Read before running — an assistant writes the same model

> *"Write a PyTorch model that classifies a padded batch of token ids with an
> embedding and a GRU, and returns two logits."*

Under-specified in exactly one place. The code it returns is the
`last_of_padding=True` branch above: it summarises at `out[:, -1, :]`, the last
position of the **padded** batch.

**Reviewer question 3: what is the shape here?** Position `MAXLEN - 1` is
padding for every review shorter than `MAXLEN` — which is most of them.

> **Prompt · ⚠ what the assistant returns**
>
> **input** · 'write a PyTorch model that classifies a padded batch of token ids with an embedding and a GRU, returning two logits'
>
> **output** · the same model summarising at `out[:, -1, :]`, and what it costs
>
> **constraint** · run it and compare — under-specified in exactly ONE place, and the code is otherwise correct

**Watch this prompt.**

* **Left open:** reviewer question 3. Position MAXLEN−1 is PADDING for every review shorter than MAXLEN, which is most of them. The model is reading the GRU's state after it has consumed a hundred zeros.
* **The usual student version:** exactly this. It is the obvious way to take 'the last output', the shapes are right, and nothing raises.
* **How you would catch it:** the prompt named the batch as padded and did not say what to do about it. A specification that mentions padding must say how the summary avoids it.

In [ ]:
torch.manual_seed(RANDOM_STATE)
padbug, curve_padbug, _ = train(
    GRUClassifier(VOCAB, last_of_padding=True), Xf_w, Lf_w, fit_y,
    Xv_w, Lv_w, val_y, tag="last-of-padding")

print(f"\ncorrect        {curve_scratch[-1]:.1%}")
print(f"last of padding {curve_padbug[-1]:.1%}")
print(f"the bug costs   {100 * (curve_scratch[-1] - curve_padbug[-1]):.2f} points")

### The test that catches it

A padding bug is exactly the class of error that a test on **invariance**
catches and a test on output shape does not: pad the same review two different
ways and the logits must not move.

> **Prompt · the test that catches it**
>
> **input** · the same review, padded two different ways
>
> **output** · whether the logits move
>
> **constraint** · test INVARIANCE, not shape — a padding bug is exactly the class of error that an output-shape test cannot see
>
> **check** · assert the correct model's logits agree to 1e-4, and print how far the buggy one moves

**Watch this prompt.**

* **Left open:** that this test generalises. Any model consuming variable-length input should be invariant to how that input was padded, and it is four lines to check.
* **The usual student version:** testing that the output shape is (N, 2), which both models pass. Shape tests catch shape bugs.
* **How you would catch it:** feed the same content two ways and require the same answer. It is the text equivalent of evaluating the same model twice.

In [ ]:
scratch.eval(); padbug.eval()
n = int(Lf_w[0])
one = scratch(Xf_w[:1, :n].to(device), torch.tensor([n]))
two = scratch(Xf_w[:1, :].to(device),  torch.tensor([n]))
assert torch.allclose(one, two, atol=1e-4), "padding is being read"
print("padding invariance holds for the correct model")

one_b = padbug(Xf_w[:1, :n].to(device), torch.tensor([n]))
two_b = padbug(Xf_w[:1, :].to(device),  torch.tensor([n]))
print(f"the assistant's model moves by "
      f"{(one_b - two_b).abs().max().item():.3f} — same review, different padding")

## 11 · The swap: change the table, and nothing else

The diagnosis is not the architecture. It is that the embedding table is being
asked to learn English from 5,000 sentiment labels.

Two changes, applied **one at a time**, so the measurement says which one did
the work.

> **Prompt · change ONE thing — the tokenizer**
>
> **input** · the same reviews, tokenised into subwords
>
> **output** · the subword batches, and a model trained on them from random vectors
>
> **constraint** · change the tokenizer and NOTHING else — same architecture, same seed, same epochs — so the measurement says which change did the work
>
> **check** · assert the batch shape before training on it

**Watch this prompt.**

* **Left open:** that the result is a NULL RESULT and not a ranking. At the deck's scale these land within a few hundredths of a point, and two single-seed numbers that close say only that the effect is smaller than the seed-to-seed spread.
* **The usual student version:** expecting the subword tokenizer to help by itself. There was little to win — the token-level OOV rate was already a few per cent — sequences are longer in pieces, and `un`, `##watch`, `##able` are three random vectors whose composition the model must learn.
* **How you would catch it:** a subword tokenizer is not a better tokenizer by itself. It is a vocabulary that someone else's pretraining can be poured into.

In [ ]:
def encode_pieces(texts):
    out  = tk(list(texts), truncation=True, max_length=MAXLEN,
              padding="max_length")
    ids  = np.asarray(out["input_ids"], dtype=np.int64)
    lens = np.asarray(out["attention_mask"], dtype=np.int64).sum(1)
    return torch.from_numpy(ids), torch.from_numpy(np.maximum(lens, 1))

Xf_s, Lf_s = encode_pieces(fit_x)
Xv_s, Lv_s = encode_pieces(val_x)
assert Xf_s.shape == (N_FIT, MAXLEN)
print(f"subword batch {tuple(Xf_s.shape)}   vocabulary {tk.vocab_size:,}")

torch.manual_seed(RANDOM_STATE)
_, curve_wp_random, _ = train(
    GRUClassifier(tk.vocab_size), Xf_s, Lf_s, fit_y, Xv_s, Lv_s, val_y,
    tag="subword / random")

At the deck's scale these two land within a few hundredths of a point of each
other — a **null result, not a ranking**, since two single-seed numbers that
close say only that the effect is smaller than the seed-to-seed spread. At this
notebook's smaller scale it is usually worse. Either way the reading is the
same:

* there was little to win — the token-level OOV rate was already a few per cent;
* sequences are longer in pieces, so the same budget of positions holds fewer
  words;
* `un`, `##watch`, `##able` are three random vectors, and the model has to learn
  that their *composition* is negative.

A subword tokenizer is not a better tokenizer by itself. It is a vocabulary that
someone else's pretraining can be poured into.

> **Prompt · pour the pretraining in**
>
> **input** · DistilBERT's 768-wide embedding table
>
> **output** · a 128-wide projection of it, rescaled to match nn.Embedding's own scale
>
> **constraint** · rescale after projecting — PCA components have the variance of the data, and dropping a table with the wrong scale into a randomly-initialised architecture changes the effective learning rate of everything downstream
>
> **check** · assert the projected shape, and report how much variance 128 components keep

**Watch this prompt.**

* **Left open:** that this is the SVD thread from application 5, used for something other than pictures. Same computation, different purpose.
* **The usual student version:** using the 768-wide table directly and widening the GRU to match, which changes two things at once and makes the comparison meaningless.
* **How you would catch it:** when you transplant a fitted object into a different architecture, match its scale to what the architecture expects. `Z / Z.std() * 0.1` is that line, and it is easy to leave out.

In [ ]:
# The same vocabulary already has a trained embedding table. It is 768 wide and
# our architecture is 128 wide, so project with PCA — thread 5, from Lecture 10.
from sklearn.decomposition import PCA
from transformers import AutoModel

bert = AutoModel.from_pretrained(BERT)
E = bert.embeddings.word_embeddings.weight.detach().numpy()[:tk.vocab_size]
print(f"pretrained table: {E.shape}")

pca = PCA(n_components=EMB_DIM, random_state=RANDOM_STATE)
Z = pca.fit_transform(E)
Z = (Z / Z.std() * 0.1).astype(np.float32)      # match nn.Embedding's own scale

assert Z.shape == (tk.vocab_size, EMB_DIM)
print(f"{EMB_DIM} components keep "
      f"{pca.explained_variance_ratio_.sum():.1%} of the variance")

> **Prompt · frozen, then tuned**
>
> **input** · the pretrained table, once held fixed and once allowed to move
>
> **output** · both validation curves
>
> **constraint** · run BOTH — frozen isolates what the pretrained vectors are worth, and tuned shows what adapting them adds

**Watch this prompt.**

* **Left open:** that only the parameters with `requires_grad` reach the optimiser. The train function filters them, which is why freezing actually freezes rather than merely zeroing gradients.
* **The usual student version:** passing `net.parameters()` to Adam with a frozen table. Adam then holds moment estimates for 3.9 million parameters it will never update, and on a small GPU that is where the memory goes.
* **How you would catch it:** two runs, one variable. Frozen against tuned is the cheapest possible ablation and it answers a question the single tuned run cannot.

In [ ]:
torch.manual_seed(RANDOM_STATE)
_, curve_frozen, _ = train(
    GRUClassifier(tk.vocab_size, init=Z, freeze=True),
    Xf_s, Lf_s, fit_y, Xv_s, Lv_s, val_y, tag="subword / frozen")

torch.manual_seed(RANDOM_STATE)
tuned, curve_tuned, _ = train(
    GRUClassifier(tk.vocab_size, init=Z, freeze=False),
    Xf_s, Lf_s, fit_y, Xv_s, Lv_s, val_y, tag="subword / tuned")

## 12 · The test set. Once.

Everything above used the fit and validation splits only. This is the first and
last time the test half is scored.

⏱ **about a minute** — encoding 25,000 reviews twice and two forward passes.

> **Prompt · the test set, once**
>
> **input** · all 25,000 test reviews, encoded both ways
>
> **output** · every configuration's test accuracy, beside both anchors
>
> **constraint** · encode with BOTH tokenizers — the word models and the subword models cannot share a test batch
>
> **check** · assert both encodings produced 25,000 rows

**Watch this prompt.**

* **Left open:** that everything above used the fit and validation splits only. This is the first and last time the test half is scored.
* **The usual student version:** scoring a subword model on the word-tokenised batch. The ids are valid indices into a different vocabulary, so it runs, and the accuracy is near chance for a reason that looks like a modelling failure.
* **How you would catch it:** keep the anchors in the final table. The bag of words is in that column for a reason, and it is not there to be flattered.

In [ ]:
Xt_w, Lt_w = encode_words(test_x)
Xt_s, Lt_s = encode_pieces(test_x)
assert Xt_w.shape[0] == 25_000 and Xt_s.shape[0] == 25_000

results = {
    "always one class":               majority,
    "tf-idf + logistic regression":   bow_acc,
    "GRU, our words, random":         accuracy(scratch, Xt_w, Lt_w, test_y),
    "GRU, subword, pretrained tuned": accuracy(tuned,   Xt_s, Lt_s, test_y),
    "GRU, the assistant's version":   accuracy(padbug,  Xt_w, Lt_w, test_y),
}
for name, acc in results.items():
    print(f"{name:34s} {acc:.1%}")

> **Prompt · four curves**
>
> **input** · every configuration's validation curve
>
> **output** · all four on one axis
>
> **constraint** · one axis, so the four are comparable, and integer epoch ticks — there are two of them and matplotlib will otherwise offer 1.5

**Watch this prompt.**

* **Left open:** that two epochs is very few, and the curves are correspondingly uninformative about what would happen next. The deck runs four.
* **The usual student version:** reading a trend from two points. The figure shows where the four configurations sit, not where they are going.
* **How you would catch it:** when you have very few points, plot markers and not just lines. A line between two points invites extrapolation that the data cannot support.

In [ ]:
plt.figure(figsize=(9, 3))
for label, c in (("our words, random", curve_scratch),
                 ("subword, random", curve_wp_random),
                 ("subword, pretrained frozen", curve_frozen),
                 ("subword, pretrained tuned", curve_tuned)):
    plt.plot(range(1, EPOCHS + 1), [100 * v for v in c], "o-", label=label)
plt.xticks(range(1, EPOCHS + 1))
plt.xlabel("epoch"); plt.ylabel("validation accuracy, %")
plt.legend(); plt.tight_layout(); plt.show()

## 12b · The requirement the metric cannot see

The brief said *"within the hour"*, which makes inference cost a stated
requirement — and accuracy says nothing about it. So time it: raw strings
in, labels out, tokenising **included**, because that is what the desk pays
per review.

A wall clock is the least reproducible number you will produce, so take the
**median of several passes** and never a single one.

⏱ **two to three minutes** — it re-tokenises the whole test set on every
pass, which is exactly the cost being measured.

> **Prompt · ⏱ 2-3 min — the requirement the metric cannot see**
>
> **input** · the full test set, three times per configuration
>
> **output** · the median seconds per pass and the milliseconds per review
>
> **constraint** · time raw strings IN and labels OUT, tokenising INCLUDED — that is what the desk pays per review — and take the MEDIAN of several passes, never a single one
>
> **check** · assert counting words is the cheapest, which is the expected ordering and worth failing loudly if it is not

**Watch this prompt.**

* **Left open:** that a wall clock is the least reproducible number you will produce. The min and max are printed beside the median so the reader can see the spread.
* **The usual student version:** timing the forward pass only, which omits tokenisation — the part that actually dominates for the tf-idf model and is not free for the others.
* **How you would catch it:** none of these is anywhere near the stated hour, so the cost column does not decide anything here. A requirement that turns out not to bind is still worth measuring: you did not know it did not bind until you measured.

In [ ]:
def score_time(fn, repeats=3):
    """Median of `repeats` full passes. The mean is dominated by whichever
    pass collided with something else on the machine."""
    runs = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        runs.append(time.perf_counter() - t0)
    return sorted(runs)[len(runs) // 2], min(runs), max(runs)

@torch.no_grad()
def rnn_pass(net, enc):
    X, L = enc(test_x)
    net.eval()
    for i in range(0, len(X), 256):
        net(X[i:i + 256].to(device), L[i:i + 256])

costs = {
    "tf-idf + logistic regression": score_time(
        lambda: bow.predict(vec.transform(test_x))),
    "GRU, our words":  score_time(lambda: rnn_pass(scratch, encode_words)),
    "GRU, subword":    score_time(lambda: rnn_pass(tuned,   encode_pieces)),
}
for name, (med, lo, hi) in costs.items():
    print(f"{name:30s} {med:6.1f} s   {1000 * med / len(test_x):5.2f} ms/review"
          f"   (min {lo:.1f}, max {hi:.1f})")

cheapest = min(costs, key=lambda k: costs[k][0])
assert cheapest.startswith("tf-idf"), f"expected counting words to win, got {cheapest}"

None of these is anywhere near an hour, so on this brief the cost column does
not decide anything. A requirement that turns out not to bind is still worth
measuring — you did not know it did not bind until you measured. It starts
to bind in the next lecture, where the model is 25 times larger.

## 13 · Where we are

Write your **best accuracy** on the same sheet of paper, next to what you
predicted. Bring it to the next lecture — we open by comparing them.

Four questions we did not answer, and all four are the next lecture:

1. What exactly does `CrossEntropyLoss` receive, and why does it want the raw
   two numbers rather than probabilities?
2. We borrowed a tokenizer and a table. What if we borrowed the whole model?
3. The desk wants complaints *grouped*, not only flagged.
4. Is the last number good? Compared with what ceiling?

Do not fix anything yet.